# Retries, Fallbacks, and Error Handling

This notebook focuses on a skill that makes agentic systems feel professional: handling failure without breaking the user experience.

A senior AI system should retry the right things, fail fast on the wrong things, and fall back to a safer path when confidence is low.


## What failure handling should cover

- tool timeout
- API rate limit
- malformed model output
- missing context
- unsafe action request
- downstream validation failure
- empty or contradictory retrieval results


In [ ]:
from dataclasses import dataclass

@dataclass
class AgentResult:
    status: str
    message: str
    fallback_used: bool = False

AgentResult(status='ok', message='primary path succeeded')


In [ ]:
def run_with_retry(task, attempts: int = 3):
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            return task(attempt)
        except Exception as exc:
            last_error = exc
    raise last_error

def flaky_task(attempt: int):
    if attempt < 2:
        raise RuntimeError('temporary failure')
    return {'status': 'ok', 'attempt': attempt}

run_with_retry(flaky_task)


## Retry tips

Good retries are selective, not blind:

- retry transient network issues
- do not retry invalid user input
- do not retry unsafe action requests
- add jitter for external APIs
- cap the total wait time
- log every retry attempt


In [ ]:
def fallback_route(message: str, confidence: float) -> AgentResult:
    if confidence < 0.6:
        return AgentResult(status='needs_review', message='Escalated to human review', fallback_used=True)
    return AgentResult(status='ok', message=message)

fallback_route('answer ready', 0.42)


## Useful enterprise patterns

- circuit breaker around flaky tools
- cached fallback answer for stable questions
- human approval for high-risk actions
- idempotency keys for write operations
- structured error payloads
- observable traces with the failure reason attached


In [ ]:
def should_escalate(error_text: str, confidence: float) -> bool:
    risky_terms = ['delete', 'drop', 'close account', 'pay']
    if any(term in error_text.lower() for term in risky_terms):
        return True
    return confidence < 0.5

should_escalate('Need to drop this customer record', 0.95)


## LangGraph angle

This is where LangGraph shines: each branch can be explicit.

You can route from the main node into:

- retry node
- validation node
- safe fallback node
- human approval node

That makes the workflow easier to reason about than a single opaque agent loop.


## Production reminders

- measure retry rate and failure rate
- watch for loops
- keep fallback answers clearly marked
- surface uncertainty to the user
- store error traces for debugging and evaluation
- test the unhappy path as part of every release


## Demo ideas

Great demonstrations for this notebook include:

- tool failure with safe fallback
- malformed JSON from an LLM with repair logic
- low-confidence retrieval with escalation
- rate-limited API with backoff and retry
